[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_10/SFM_ch10_var_methods/SFM_ch10_var_methods.ipynb)

# SFM_ch10_var_methods

VaR 1% and ES 2.5% of daily log returns (VaR_alpha = -q_alpha, the loss exceeded with probability alpha) by historical simulation, Normal, Student-t (maximum likelihood), Cornish-Fisher and EVT (peaks over threshold, GPD above the 90% quantile of the losses) for the S&P 500, DAX, BET, Bitcoin, Banca Transilvania and OMV Petrom (as in the SFE Quantlets VaRest and SFEVaRqqplot); VaR as a function of the tail probability from 5% to 0.1%; bootstrap intervals of the historical VaR and ES; the square-root-of-time rule against 10-day historical VaR; GARCH(1,1)-t and filtered historical simulation VaR for the next day.

*Statistics of Financial Markets (SFM), Chapter 10: VaR, ES and backtesting. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_10'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# the arch package (maximum-likelihood estimation of GARCH models) is not part of Google Colab
!pip install -q arch

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Analysis

In [ ]:
from arch import arch_model

NAME = {'sp500': 'S&P 500', 'dax': 'DAX', 'bet': 'BET', 'btc': 'Bitcoin', 'tlv': 'Banca Transilvania', 'snp': 'OMV Petrom', 'brd': 'BRD'}

START = {'sp500': '2000-01-01', 'dax': '2000-01-01', 'bet': '2000-01-01', 'btc': None, 'tlv': '2010-01-01', 'snp': '2010-01-01', 'brd': '2010-01-01'}

ASSETS = ['sp500', 'dax', 'bet', 'btc', 'tlv', 'snp']

BT_ASSETS = ['sp500', 'dax', 'bet', 'btc']

BAD_DAYS = {'tlv': ['2016-05-30', '2016-05-31']}

COLORS = {'sp500': '#1A3A6E', 'dax': '#17A2B8', 'bet': '#CD0000', 'btc': '#B5853F', 'tlv': '#E67E22', 'snp': '#2E7D32'}

EPISODES = {'2008': ('2008-09-01', '2009-03-31'), '2020': ('2020-02-15', '2020-05-31')}

ALPHA_VAR = 0.01

ALPHA_ES = 0.025

WINDOW = 500

REFIT = 250

OOS_START = {'sp500': '2005-01-01', 'dax': '2005-01-01', 'bet': '2005-01-01', 'btc': '2017-01-01'}

POT_Q = 0.9

METHODS = ['HS', 'Normal', 'GARCH-t', 'FHS']

MCOL = {'HS': '#B5853F', 'Normal': '#2E7D32', 'GARCH-t': '#CD0000', 'FHS': '#1A3A6E'}

PORT = ('bet', 'sp500')

WEIGHTS = (0.5, 0.5)

N_SIM = 200000

N_Z2 = 1000

SEED = 2026

PLUS = {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 5: 0.4, 6: 0.5, 7: 0.65, 8: 0.75, 9: 0.85}

def returns(k, start=None, end=None):
    """Daily log returns in % on the series' own calendar; known data errors removed."""
    r = log_returns(k, start or START.get(k)) if end is None else log_returns(k, start or START.get(k), end)
    if k in BAD_DAYS:
        r = r.drop(pd.to_datetime(BAD_DAYS[k]), errors='ignore')
    return r


def hs_var(x, a=ALPHA_VAR):
    """Historical simulation: VaR_a = -r_(k), the k-th smallest return, k = ceil(n a)."""
    xs = np.sort(np.asarray(x, float))
    k = int(np.ceil(len(xs) * a - 1e-9))
    return float(-xs[k - 1])


def hs_es(x, a=ALPHA_ES):
    """Historical simulation: ES_a = minus the average of the k smallest returns, k = ceil(n a)."""
    xs = np.sort(np.asarray(x, float))
    k = int(np.ceil(len(xs) * a - 1e-9))
    return float(-xs[:k].mean())


def normal_var(mu, sd, a=ALPHA_VAR):
    """VaR_a = -(mu + sd z_a), z_a the a-quantile of N(0, 1) (negative)."""
    return float(-(mu + sd * stats.norm.ppf(a)))


def normal_es(mu, sd, a=ALPHA_ES):
    """ES_a = -mu + sd phi(z_a) / a."""
    return float(-mu + sd * stats.norm.pdf(stats.norm.ppf(a)) / a)


def t_es_factor(nu, a=ALPHA_ES):
    """ES of a Student-t variable with nu degrees of freedom (scale 1): f(t_a) (nu + t_a^2) / ((nu - 1) a)."""
    q = stats.t.ppf(a, nu)
    return float(stats.t.pdf(q, nu) * (nu + q ** 2) / ((nu - 1) * a))


def std_t_q(nu, a=ALPHA_VAR):
    """a-quantile of the standardised Student-t (variance 1): t_nu^{-1}(a) sqrt((nu - 2)/nu)."""
    return float(stats.t.ppf(a, nu) * np.sqrt((nu - 2) / nu))


def std_t_es(nu, a=ALPHA_ES):
    """ES_a of the standardised Student-t (variance 1)."""
    return float(np.sqrt((nu - 2) / nu) * t_es_factor(nu, a))


def t_fit(x):
    """Student-t with location m, scale s and nu degrees of freedom, fitted by maximum likelihood (scipy)."""
    nu, m, s = stats.t.fit(np.asarray(x, float))
    return float(nu), float(m), float(s)


def t_var_es(x, a_var=ALPHA_VAR, a_es=ALPHA_ES):
    nu, m, s = t_fit(x)
    return {'var': float(-(m + s * stats.t.ppf(a_var, nu))), 'es': float(-m + s * t_es_factor(nu, a_es)),
            'nu': nu, 'm': m, 's': s}


def cf_z(a, S, K):
    """Cornish-Fisher quantile of a standardised variable with skewness S and excess kurtosis K."""
    z = stats.norm.ppf(a)
    return z + (z ** 2 - 1) * S / 6 + (z ** 3 - 3 * z) * K / 24 - (2 * z ** 3 - 5 * z) * S ** 2 / 36


def cf_var_es(x, a_var=ALPHA_VAR, a_es=ALPHA_ES):
    """Cornish-Fisher VaR = -(mu + sd z_cf); ES = minus the average of the CF quantiles on (0, a_es)."""
    x = np.asarray(x, float)
    mu, sd = x.mean(), x.std(ddof=1)
    S, K = float(stats.skew(x)), float(stats.kurtosis(x))
    u = (np.arange(2000) + 0.5) / 2000 * a_es
    return {'var': float(-(mu + sd * cf_z(a_var, S, K))), 'es': float(-(mu + sd * cf_z(u, S, K).mean())),
            'z': float(cf_z(a_var, S, K)), 'S': S, 'K': K, 'mu': float(mu), 'sd': float(sd)}


def pot_fit(x, q=POT_Q):
    """GPD fit to the losses L = -r above the threshold u = q-quantile of the losses (peaks over threshold)."""
    L = -np.asarray(x, float)
    u = float(np.quantile(L, q))
    exc = L[L > u] - u
    xi, _, beta = stats.genpareto.fit(exc, floc=0)
    return {'u': u, 'xi': float(xi), 'beta': float(beta), 'Nu': int(len(exc)), 'n': int(len(L))}


def pot_var(f, a):
    """VaR_a = u + beta/xi [(n a / N_u)^(-xi) - 1], valid for a < N_u / n."""
    return float(f['u'] + f['beta'] / f['xi'] * ((f['n'] * a / f['Nu']) ** (-f['xi']) - 1))


def pot_es(f, a):
    """ES_a = VaR_a / (1 - xi) + (beta - xi u) / (1 - xi), valid for xi < 1."""
    return float(pot_var(f, a) / (1 - f['xi']) + (f['beta'] - f['xi'] * f['u']) / (1 - f['xi']))


def garch_fit(r, last_obs=None):
    """GARCH(1,1) with constant mean and standardised Student-t innovations (arch package)."""
    am = arch_model(r, mean='Constant', vol='GARCH', p=1, q=1, dist='t')
    return am.fit(disp='off', last_obs=last_obs, options={'maxiter': 2000})


def conditional_next_day(k, a_var=ALPHA_VAR, a_es=ALPHA_ES):
    """VaR and ES for the day after the last observation: GARCH(1,1)-t and filtered historical simulation."""
    r = returns(k)
    res = garch_fit(r)
    p = res.params
    mu, nu = float(p['mu']), float(p['nu'])
    s_next = float(np.sqrt(res.forecast(horizon=1, reindex=False).variance.values[-1, 0]))
    z = res.std_resid.dropna().values
    return {'mu': mu, 'nu': nu, 'omega': float(p['omega']), 'alpha': float(p['alpha[1]']), 'beta': float(p['beta[1]']),
            'sigma': s_next, 'q_t': std_t_q(nu, a_var), 'es_t': std_t_es(nu, a_es),
            'var_g': float(-(mu + s_next * std_t_q(nu, a_var))), 'es_g': float(-mu + s_next * std_t_es(nu, a_es)),
            'q_z': float(-hs_var(z, a_var)), 'es_z': hs_es(z, a_es),
            'var_f': float(-(mu + s_next * -hs_var(z, a_var))), 'es_f': float(-mu + s_next * hs_es(z, a_es)),
            'n': int(len(r)), 'last': r.index[-1].date().isoformat(), 'r_last': float(r.iloc[-1])}


def methods_table(assets=ASSETS):
    """VaR 1% and ES 2.5% (in %) of each series on its whole sample by five unconditional methods, plus the
    conditional GARCH-t and FHS values for the next day."""
    out = {}
    for k in assets:
        x = returns(k)
        mu, sd = float(x.mean()), float(x.std())
        tt = t_var_es(x)
        cf = cf_var_es(x)
        f = pot_fit(x)
        c = conditional_next_day(k)
        out[k] = {'n': int(len(x)), 'first': x.index[0].date().isoformat(), 'mu': mu, 'sd': sd,
                  'skew': cf['S'], 'exkurt': cf['K'], 'nu': tt['nu'], 'xi': f['xi'],
                  'HS': {'var': hs_var(x), 'es': hs_es(x)},
                  'Normal': {'var': normal_var(mu, sd), 'es': normal_es(mu, sd)},
                  'Student-t': {'var': tt['var'], 'es': tt['es']},
                  'Cornish-Fisher': {'var': cf['var'], 'es': cf['es']},
                  'EVT': {'var': pot_var(f, ALPHA_VAR), 'es': pot_es(f, ALPHA_ES)},
                  'GARCH-t': {'var': c['var_g'], 'es': c['es_g']}, 'FHS': {'var': c['var_f'], 'es': c['es_f']},
                  'pot': f, 'cf_z': cf['z'], 't': tt, 'cond': c}
    return out


def fig_var_es_def(k='sp500', save=True):
    """Histogram of daily returns with minus VaR 1% and minus ES 2.5% (historical simulation)."""
    x = returns(k)
    v1, q25, e25 = hs_var(x, ALPHA_VAR), -hs_var(x, ALPHA_ES), hs_es(x, ALPHA_ES)
    fig, ax = plt.subplots(figsize=(10, 4.0))
    bins = np.arange(-13, 12.01, 0.25)
    ax.hist(x, bins=bins, density=True, color=st.MainBlue, alpha=0.75, label=f'daily log returns of the {NAME[k]} (%)')
    ax.axvspan(-13, q25, color=st.IDAred, alpha=0.08, label='worst 2.5% of the days')
    ax.axvline(-v1, color=st.IDAred, lw=2, label=f'minus VaR 1% = {-v1:.2f}%')
    ax.axvline(-e25, color=st.Purple, lw=2, ls='--', label=f'minus ES 2.5% = {-e25:.2f}%')
    ax.set_xlim(-13, 12)
    ax.set_xlabel('daily return (%)')
    ax.set_ylabel('density')
    st.legend_outside_bottom(ax, ncol=2, y=-0.2)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch10_var_es_def')
    return {'var1': v1, 'q25': q25, 'es25': e25, 'n': int(len(x)), 'n_tail1': int((x < -v1).sum()),
            'min': float(x.min()), 'date_min': x.idxmin().date().isoformat()}


def var_curve(x, alphas):
    """VaR_alpha (positive, in %) for each tail probability by five unconditional methods."""
    mu, sd = float(np.mean(x)), float(np.std(x, ddof=1))
    tt = t_var_es(x)
    cf = cf_var_es(x)
    f = pot_fit(x)
    out = {'historical': [hs_var(x, a) for a in alphas], 'Normal': [normal_var(mu, sd, a) for a in alphas],
           'Student-t': [float(-(tt['m'] + tt['s'] * stats.t.ppf(a, tt['nu']))) for a in alphas],
           'Cornish-Fisher': [float(-(mu + sd * cf_z(a, cf['S'], cf['K']))) for a in alphas],
           'EVT (POT)': [pot_var(f, a) if a < f['Nu'] / f['n'] else np.nan for a in alphas]}
    return out


def fig_var_curves(keys=('sp500', 'bet'), save=True):
    """VaR as a function of the tail probability alpha (log axis) for the S&P 500 and the BET."""
    alphas = np.array([0.001, 0.002, 0.003, 0.005, 0.0075, 0.01, 0.015, 0.02, 0.025, 0.03, 0.04, 0.05])
    sty = {'historical': (st.DarkText, 'o', '-'), 'Normal': (st.Forest, None, '--'), 'Student-t': (st.MainBlue, None, '-'),
           'Cornish-Fisher': (st.Purple, None, '-.'), 'EVT (POT)': (st.IDAred, None, '-')}
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
    out = {}
    for ax, k in zip(axes, keys):
        x = returns(k).values
        cur = var_curve(x, alphas)
        for lab, v in cur.items():
            c, m, ls = sty[lab]
            ax.plot(100 * alphas, v, color=c, marker=m, ms=4, ls=ls if m is None else 'none', lw=1.6, label=lab)
        ax.set_xscale('log')
        ax.set_xticks([0.1, 0.25, 0.5, 1, 2.5, 5])
        ax.set_xticklabels(['0.1', '0.25', '0.5', '1', '2.5', '5'])
        ax.invert_xaxis()
        ax.set_xlabel('tail probability alpha (%)')
        ax.set_ylabel('VaR (%)')
        ax.set_title(NAME[k])
        out[k] = {lab: dict(zip([f'{a:g}' for a in alphas], map(float, v))) for lab, v in cur.items()}
    st.fig_legend_bottom(fig, ncol=5, y=0.0)
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch10_var_curves')
    return out


def precision(k='sp500', n_last=500, B=2000, seed=SEED):
    """Bootstrap (Chapter 2) 95% intervals of the historical VaR 1% and ES 2.5%: whole sample and last n_last days."""
    rng = np.random.default_rng(seed)
    x = returns(k).values
    out = {}
    for lab, y in [('all', x), ('last', x[-n_last:])]:
        idx = rng.integers(0, len(y), (B, len(y)))
        bv = np.array([hs_var(y[i]) for i in idx])
        be = np.array([hs_es(y[i]) for i in idx])
        out[lab] = {'n': int(len(y)), 'var': hs_var(y), 'es': hs_es(y), 'var_lo': float(np.quantile(bv, 0.025)),
                    'var_hi': float(np.quantile(bv, 0.975)), 'es_lo': float(np.quantile(be, 0.025)),
                    'es_hi': float(np.quantile(be, 0.975)), 'var_se': float(bv.std()), 'es_se': float(be.std())}
    return out


def horizon_check(k='sp500', h=10):
    """Square-root-of-time rule: sqrt(h) x one-day historical VaR 1% against the historical VaR 1% of overlapping
    h-day log returns."""
    r = returns(k)
    rh = r.rolling(h).sum().dropna()
    return {'var1': hs_var(r), 'var_sqrt': float(np.sqrt(h) * hs_var(r)), 'var_h': hs_var(rh.values), 'h': h,
            'var_n1': normal_var(r.mean(), r.std()), 'var_nh': normal_var(h * r.mean(), np.sqrt(h) * r.std())}

## Run

In [ ]:
print(fig_var_es_def())
MT = methods_table()
print(pd.DataFrame({(NAME[k], m): {'VaR 1%': MT[k][m]['var'], 'ES 2.5%': MT[k][m]['es']} for k in ASSETS for m in ['HS', 'Normal', 'Student-t', 'Cornish-Fisher', 'EVT', 'GARCH-t', 'FHS']}).T.round(2))
fig_var_curves()
print(precision())
print(horizon_check())

![sfm_ch10_var_es_def](./sfm_ch10_var_es_def.png)

Output: `sfm_ch10_var_es_def.pdf`

![sfm_ch10_var_curves](./sfm_ch10_var_curves.png)

Output: `sfm_ch10_var_curves.pdf`